# Per-prompt cost comparison

This notebook reads `result_1_*.json` and plots per-prompt utility curves for fixed-n versus Pandora's Box.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

llm_mapping = {
    'gemma2_9b': 'google/gemma-2-9b-it',
    'llama3.1_8b': 'meta-llama/Llama-3.1-8B-Instruct',
    'llama3.2_3b': 'meta-llama/Llama-3.2-3B-Instruct',
    'mistral_7b': 'mistralai/Mistral-7B-Instruct-v0.3',
    'qwen2.5_7b': 'Qwen/Qwen2.5-7B-Instruct',
    'qwen3_4b': 'Qwen/Qwen3-4B-Instruct-2507',
}

rm_mapping = {
    'fsfairx_rm': 'sfairXC/FsfairX-LLaMA3-RM-v0.1',
    'mistral_rm': 'weqweasdas/RM-Mistral-7B',
}


In [ ]:
# Config
dataset = 'alpaca'
rm = 'fsfairx_rm'
llm = 'mistral_7b'
distribution = 'shifted_exponential'
transformation = 'cdf'
batch_size = '1'
alpha = '0.99'

result_path = Path(f'../slurm_result_{dataset}/result_1_{rm}_{llm}_{distribution}_{transformation}_bs{batch_size}_a{alpha}.json')
with result_path.open() as f:
    data = json.load(f)

print(f'Loaded: {result_path}')
print(f'Prompts: {len(data["per_prompt"])}')


In [ ]:
# Choose prompt index and which costs (by index in data['costs']) to plot
prompt_index = 0
cost_indices = [0, 3, 6, 9]  # adjust as needed


In [ ]:
def plot_revenue_comparison_from_data(data, prompt_index, cost_indices, llm_name=None, rm_name=None, dataset_name=None):
    prompt = data['per_prompt'][prompt_index]
    costs = data['costs']

    llm_label = llm_mapping.get(llm_name, llm_name)
    rm_label = rm_mapping.get(rm_name, rm_name)

    fig, axes = plt.subplots(1, len(cost_indices), figsize=(5 * len(cost_indices), 5))
    if len(cost_indices) == 1:
        axes = [axes]

    fig.text(0.5, 0.98, 'Profit Comparison: Fixed-n vs Pandora's Box',
             fontsize=16, fontweight='bold', ha='center', va='top')
    subtitle = f'LLM: {llm_label}  |  Reward Model: {rm_label} | Dataset: {dataset_name}'
    fig.text(0.5, 0.92, subtitle, fontsize=13, ha='center', va='top')

    for ax, cost_idx in zip(axes, cost_indices):
        cost = costs[cost_idx]
        fixed_entry = prompt['fixed_n'][cost_idx]
        pandora_entry = prompt['pandora'][cost_idx]

        xs = [item['n'] for item in fixed_entry['results']]
        ys = [item['mean_utility'] for item in fixed_entry['results']]
        adaptive = pandora_entry['median_utility']

        ax.scatter(xs, ys, s=20, c='blue', alpha=0.7, label='Fixed-n')
        ax.plot(xs, ys, 'b-', alpha=0.3, linewidth=1)
        ax.axhline(y=adaptive, color='red', linestyle='--', linewidth=2, label='Pandora's Box')

        max_na = float(np.max(ys))
        optimal_samples = xs[int(np.argmax(ys))]
        ax.scatter([optimal_samples], [max_na], s=100, c='green', marker='*', zorder=5,
                   label='Optimal Fixed-n')

        ax.set_title(f'Cost = {cost}', fontsize=12)
        ax.set_xlabel('Sample Count', fontsize=12)
        ax.grid(True, alpha=0.3, linestyle=':', linewidth=0.5)
        ax.set_facecolor('#f9f9f9')
        if ax is axes[0]:
            ax.set_ylabel('Utility (Win Rate - Cost * n)', fontsize=12)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='center left', bbox_to_anchor=(0.98, 0.5),
               ncol=1, frameon=True, fontsize=12, title='Algorithm')
    plt.tight_layout(rect=[0, 0, 0.97, 0.88])
    plt.show()
    return fig


In [ ]:
plot_revenue_comparison_from_data(
    data,
    prompt_index=prompt_index,
    cost_indices=cost_indices,
    llm_name=llm,
    rm_name=rm,
    dataset_name=dataset,
)
